# ResNet50 mediante Transfer Learning

**Proyecto:** clasificación de defectos superficiales en palta Hass.  
**Estado:** preparación para una etapa posterior de entrenamiento.

La entrega actual corresponde a `5. feature_engineering/feature_engineering_palta.ipynb`.
Este notebook conserva la preparación del clasificador, los pesos de clase, callbacks,
curvas y métricas del trabajo anterior. Las funciones de entrenamiento y evaluación
quedan definidas, sin llamadas automáticas. **Ejecutar todo no entrena, no descarga
pesos y no abre imágenes de test.** La inspección opcional de un lote está desactivada.

**Trazabilidad:** este archivo procede de `4. entrenamiento/02_Baseline.ipynb`, que
utilizaba MobileNetV3Small. Se adaptó a **ResNet50** y se conservó el trabajo reutilizable.
La arquitectura principal del proyecto es ResNet50 mediante Transfer Learning.


## 1. Configuración

Cada archivo representa una palta con tres franjas de cámaras. El pipeline conserva
la imagen completa y ajusta su tamaño a **256 × 512 píxeles (alto × ancho)** con relleno,
sin recorte ni deformación. Es una propuesta inicial; la conservación de defectos
pequeños debe revisarse con las imágenes reales antes de fijar esta resolución.

`utilidades_imagenes.py` comparte la lectura RGB, la compatibilidad TIFF y el ajuste
de tamaño con Feature Engineering. Los tensores llegan al modelo en RGB, `float32`,
escala 0–255. El preprocesamiento específico de ResNet50 se aplica dentro del modelo.


In [1]:
from pathlib import Path
import sys
import hashlib
import json
import random

import numpy as np
import pandas as pd

# Encontrar la raíz desde el notebook o desde el repositorio.
RAIZ = next((ruta for ruta in [Path.cwd(), *Path.cwd().parents]
             if (ruta / "README.md").is_file()
             and (ruta / "utilidades_imagenes.py").is_file()), None)
if RAIZ is None:
    raise FileNotFoundError("Abra el notebook desde la carpeta del proyecto.")
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from utilidades_imagenes import (
    encontrar_raiz, resolver_dataset, listar_imagenes, crear_dataset_tf,
)

RAIZ = encontrar_raiz()
RUTA_DATASET_RELATIVA = None  # Detectar DATASET_PALTA; si hay dos copias, indicar una ruta relativa.
SALIDA = RAIZ / "7. resultados" / "entrenamiento"
SEMILLA = 42
TAMANO_ENTRADA = (256, 512)
TAMANO_LOTE = 8
INSPECCIONAR_LOTE = False

random.seed(SEMILLA)
np.random.seed(SEMILLA)
print("ResNet50 preparado para una etapa posterior. Entrenamiento sin ejecutar.")


ResNet50 preparado para una etapa posterior. Entrenamiento sin ejecutar.


## 2. Datos de entrenamiento y validación

Las etiquetas se definen una vez a partir de las clases de train y se reutilizan en
validation y, al finalizar el experimento, en test. La palta completa sigue siendo
la unidad de observación. No se separan las franjas en muestras independientes.

Los archivos originales se leen desde sus particiones actuales. El notebook no
mueve, renombra ni sobrescribe imágenes.


In [2]:
def preparar_datos():
    dataset = resolver_dataset(RAIZ, RUTA_DATASET_RELATIVA)
    tabla_train = listar_imagenes(dataset, "train")
    if tabla_train.empty:
        raise ValueError("Train no contiene imágenes.")
    clases = sorted(tabla_train["clase"].unique().tolist())
    if len(clases) < 2:
        raise ValueError("Se necesitan al menos dos clases en train.")
    mapa_clases = {clase: indice for indice, clase in enumerate(clases)}
    tabla_train["etiqueta"] = tabla_train["clase"].map(mapa_clases).astype("int32")
    tabla_val = listar_imagenes(dataset, "validation", clases=clases)
    if tabla_val.empty or set(tabla_val["clase"]) != set(clases):
        raise ValueError("Validation debe contener las mismas clases de train.")
    return dataset, tabla_train, tabla_val, clases


def calcular_pesos(tabla_train, clases):
    from sklearn.utils.class_weight import compute_class_weight

    etiquetas = tabla_train["etiqueta"].to_numpy(dtype=np.int32)
    valores = np.arange(len(clases), dtype=np.int32)
    pesos = compute_class_weight("balanced", classes=valores, y=etiquetas)
    return {int(clase): float(peso) for clase, peso in zip(valores, pesos)}


### Inspección opcional de un lote

Al activar `INSPECCIONAR_LOTE` se carga un lote de train, sin aumento ni
preprocesamiento de ResNet50. Sirve para comprobar forma, rango y etiquetas.
No construye una red ni consulta imágenes de test.


In [3]:
if INSPECCIONAR_LOTE:
    import matplotlib.pyplot as plt

    dataset, tabla_train, tabla_val, clases = preparar_datos()
    datos_revision = crear_dataset_tf(
        tabla_train, dataset, tamano=TAMANO_ENTRADA,
        tamano_lote=TAMANO_LOTE, entrenamiento=False, semilla=SEMILLA,
    )
    imagenes, etiquetas = next(iter(datos_revision))
    print("Forma:", tuple(imagenes.shape), "Tipo:", imagenes.dtype.name)
    print("Rango RGB:", float(np.min(imagenes)), float(np.max(imagenes)))
    print("Clases:", {indice: clase for indice, clase in enumerate(clases)})
    plt.figure(figsize=(10, 5))
    plt.imshow(imagenes[0].numpy().astype(np.uint8))
    plt.title(clases[int(etiquetas[0])])
    plt.axis("off")
    plt.show()
else:
    print("Inspección de lote desactivada.")


Inspección de lote desactivada.


## 3. Condición previa: auditoría de datos

Los reportes conservados en `7. resultados/eda/` registran **dos pares exactos entre
particiones**, uno de ellos con etiquetas distintas. El entrenamiento permanece
bloqueado mientras esos problemas figuren en la auditoría.

La función siguiente exige los reportes del EDA, comprueba su coherencia y verifica
la firma contra los archivos locales. No basta con cambiar una variable a `True`.
La verificación de hashes se ejecutaría dentro del entrenamiento futuro; incluye
integridad de test, sin obtener predicciones ni ajustar decisiones con sus métricas.

Los hashes no demuestran independencia por fruto o lote. Esa revisión sigue pendiente
y debe documentarse antes del entrenamiento. Resolver las incidencias y revisar las
particiones corresponde a una etapa posterior; esta entrega conserva el dataset.


In [4]:
def hash_archivo(ruta):
    calculo = hashlib.sha256()
    with ruta.open("rb") as archivo:
        for bloque in iter(lambda: archivo.read(1024 * 1024), b""):
            calculo.update(bloque)
    return calculo.hexdigest()


def verificar_auditoria(dataset):
    carpeta = RAIZ / "7. resultados" / "eda"
    nombres = ("eda_resumen.json", "eda_inventario.csv", "eda_duplicados.csv")
    if not all((carpeta / nombre).is_file() for nombre in nombres):
        raise FileNotFoundError("Faltan reportes de auditoría. Ejecute primero el EDA.")
    resumen = json.loads((carpeta / nombres[0]).read_text(encoding="utf-8-sig"))
    inventario = pd.read_csv(carpeta / nombres[1], encoding="utf-8-sig", keep_default_na=False)
    duplicados = pd.read_csv(carpeta / nombres[2], encoding="utf-8-sig", keep_default_na=False)
    claves = {"sha256_dataset", "archivos", "imagenes", "corruptas",
              "pares_exactos_entre_particiones", "pares_exactos_etiquetas_distintas"}
    if not claves.issubset(resumen):
        raise ValueError("El resumen del EDA está incompleto.")
    if not {"archivo", "sha256"}.issubset(inventario.columns):
        raise ValueError("El inventario no contiene rutas y hashes.")
    requeridas = {"tipo", "archivo", "archivo_coincidente", "particion",
                  "particion_coincidente", "entre_particiones", "etiquetas_distintas"}
    if not requeridas.issubset(duplicados.columns):
        raise ValueError("El reporte de duplicados está incompleto.")
    if inventario.empty or inventario["archivo"].duplicated().any():
        raise ValueError("El inventario está vacío o contiene rutas repetidas.")
    if len(inventario) != resumen["archivos"] or resumen["imagenes"] < 1:
        raise ValueError("Los conteos del EDA no coinciden con el inventario.")

    exactos = duplicados[duplicados["tipo"].isin(["bytes_exactos", "pixeles_exactos"])]
    fugas_csv = exactos["particion"].ne(exactos["particion_coincidente"]).any()
    conflictos_csv = exactos["etiquetas_distintas"].astype(str).str.lower().isin(["true", "1"]).any()
    if (resumen["pares_exactos_entre_particiones"] != 0 or fugas_csv
            or resumen["pares_exactos_etiquetas_distintas"] != 0 or conflictos_csv):
        raise RuntimeError("EDA con fugas o etiquetas contradictorias. Entrenamiento bloqueado.")
    if resumen["corruptas"] != 0:
        raise RuntimeError("El EDA registra imágenes corruptas. Revise la auditoría.")

    rutas = inventario["archivo"].tolist()
    if any(Path(ruta).is_absolute() or ".." in Path(ruta).parts for ruta in rutas):
        raise ValueError("El inventario debe contener rutas relativas dentro del dataset.")
    firma_reporte = hashlib.sha256(
        "\n".join(inventario["archivo"] + ":" + inventario["sha256"]).encode()
    ).hexdigest()
    if firma_reporte != resumen["sha256_dataset"]:
        raise ValueError("La firma del resumen no coincide con el inventario.")
    archivos_locales = {ruta.relative_to(dataset).as_posix()
                        for ruta in dataset.rglob("*") if ruta.is_file()}
    if archivos_locales != set(rutas):
        raise RuntimeError("El contenido del dataset cambió. Actualice el EDA.")
    for registro in inventario.itertuples(index=False):
        if hash_archivo(dataset / registro.archivo) != registro.sha256:
            raise RuntimeError(f"Archivo distinto al auditado: {registro.archivo}")
    return resumen


## 4. Arquitectura preparada

**Imagen RGB → aumento horizontal en train → preprocesamiento ResNet50 → extractor
congelado → Dropout → clasificación.**

La preparación usa `tf.keras.applications.resnet50.preprocess_input` una sola vez:
convierte RGB a BGR y resta las medias de ImageNet. No se divide entre 255 antes de
esta operación. Con `include_top=False`, Keras admite una entrada rectangular con
3 canales y dimensiones de al menos 32 píxeles; véase la
[documentación oficial de ResNet50](https://keras.io/api/applications/resnet/resnet_models/).

El extractor permanece con `trainable=False` y se llama con `training=False` para
mantener sus estadísticas de Batch Normalization. Se propone sólo volteo horizontal,
que conserva el orden vertical de las cámaras. Su pertinencia visual debe revisarse
antes del experimento. No se aplican recortes, volteos verticales ni cambios de color
automáticos. La [guía de Transfer Learning de Keras](https://keras.io/guides/transfer_learning/)
describe el uso del extractor congelado.

La función carga pesos ImageNet únicamente cuando se la invoca. Su definición no
crea el modelo ni inicia una descarga.


In [5]:
def construir_modelo(numero_clases, tamano=TAMANO_ENTRADA):
    import tensorflow as tf
    from tensorflow.keras.applications.resnet50 import preprocess_input

    if numero_clases < 2:
        raise ValueError("El clasificador necesita al menos dos clases.")
    tf.keras.utils.set_random_seed(SEMILLA)
    entrada = tf.keras.Input(shape=(*tamano, 3), dtype="float32", name="imagen_rgb")
    aumento = tf.keras.layers.RandomFlip("horizontal", seed=SEMILLA, name="aumento_train")
    x = aumento(entrada)
    x = tf.keras.layers.Lambda(preprocess_input, name="preprocesamiento_resnet50")(x)
    modelo_base = tf.keras.applications.ResNet50(
        input_shape=(*tamano, 3), include_top=False,
        weights="imagenet", pooling="avg",
    )
    modelo_base.trainable = False
    x = modelo_base(x, training=False)
    x = tf.keras.layers.Dropout(0.35, seed=SEMILLA)(x)
    salida = tf.keras.layers.Dense(numero_clases, activation="softmax", name="clase")(x)
    modelo = tf.keras.Model(entrada, salida, name="resnet50_palta")
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return modelo


## 5. Entrenamiento futuro

Configuración inicial heredada del trabajo anterior: lote de 8, Adam con tasa
0,001, hasta 35 épocas, Early Stopping con paciencia 7 y reducción de tasa con
paciencia 3. Los pesos de clase se calculan exclusivamente con train.

Esta primera etapa ajustaría solamente el clasificador superior. El ajuste fino
de capas de ResNet50 se decidirá posteriormente. La accuracy permite seguir la
optimización; el informe final incluye macro-F1, métricas por clase y balanced accuracy.

La función no se invoca en el notebook. Antes de usarla se requiere resolver las
incidencias del EDA y revisar identidad por fruto/lote. Guarda resultados en una
carpeta nueva por experimento, sin sobrescribir ejecuciones anteriores. El checkpoint
guarda pesos para poder reconstruir el modelo con la misma función y configuración.


In [6]:
def entrenar_resnet50(nombre_ejecucion, epocas=35):
    if Path(nombre_ejecucion).name != nombre_ejecucion or nombre_ejecucion in ("", ".", ".."):
        raise ValueError("Use un nombre simple para la carpeta de resultados.")
    if epocas < 1:
        raise ValueError("El número de épocas debe ser positivo.")
    dataset, tabla_train, tabla_val, clases = preparar_datos()
    auditoria = verificar_auditoria(dataset)

    import tensorflow as tf

    carpeta = SALIDA / nombre_ejecucion
    carpeta.mkdir(parents=True, exist_ok=False)
    datos_train = crear_dataset_tf(
        tabla_train, dataset, tamano=TAMANO_ENTRADA, tamano_lote=TAMANO_LOTE,
        entrenamiento=True, semilla=SEMILLA,
    )
    datos_val = crear_dataset_tf(
        tabla_val, dataset, tamano=TAMANO_ENTRADA, tamano_lote=TAMANO_LOTE,
        entrenamiento=False, semilla=SEMILLA,
    )
    modelo = construir_modelo(len(clases))
    pesos = calcular_pesos(tabla_train, clases)
    configuracion = {
        "modelo": "ResNet50", "clases": clases, "tamano_entrada": list(TAMANO_ENTRADA),
        "semilla": SEMILLA, "tamano_lote": TAMANO_LOTE, "epocas_maximas": epocas,
        "sha256_dataset": auditoria["sha256_dataset"], "pesos_clases": pesos,
        "preprocesamiento": "RGB 0-255; resnet50.preprocess_input dentro del modelo",
    }
    (carpeta / "configuracion.json").write_text(
        json.dumps(configuracion, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    checkpoint = carpeta / "mejor_resnet50.weights.h5"
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint(str(checkpoint), monitor="val_loss",
                                           save_best_only=True, save_weights_only=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3),
    ]
    historial = modelo.fit(
        datos_train, validation_data=datos_val, epochs=epocas,
        class_weight=pesos, callbacks=callbacks,
    )
    modelo.load_weights(str(checkpoint))
    pd.DataFrame(historial.history).to_csv(carpeta / "historial.csv", index=False)
    return {"modelo": modelo, "historial": historial, "carpeta": carpeta,
            "clases": clases, "tamano_entrada": TAMANO_ENTRADA,
            "sha256_dataset": auditoria["sha256_dataset"]}


## 6. Curvas de entrenamiento

Las curvas se calculan a partir de una ejecución real. No hay métricas ni resultados
simulados en este notebook.


In [7]:
def graficar_curvas(historial):
    import matplotlib.pyplot as plt

    figura, ejes = plt.subplots(1, 2, figsize=(11, 4))
    for eje, metrica, titulo in zip(ejes, ["loss", "accuracy"], ["Pérdida", "Accuracy"]):
        valores = historial.history
        epocas = np.arange(1, len(valores[metrica]) + 1)
        eje.plot(epocas, valores[metrica], label="Train")
        eje.plot(epocas, valores[f"val_{metrica}"], label="Validation")
        eje.set(xlabel="Época", ylabel=titulo, title=titulo)
        eje.grid(alpha=0.3)
        eje.legend()
    figura.tight_layout()
    plt.show()
    return figura


## 7. Evaluación final explícita

Test se reserva para evaluar el modelo y el protocolo ya seleccionados. Esta función
se usaría después de terminar las decisiones con train y validation. No participa en
la selección de transformaciones, pesos de clase, callbacks ni hiperparámetros.

Se calculan precisión, recall y F1 por clase, macro-F1, balanced accuracy y matriz de
confusión. Se conservan el orden de clases y la correspondencia entre archivo,
etiqueta y predicción. `entrenamiento=False` evita barajar test; la llamada `predict`
desactiva el aumento de datos del modelo.


In [8]:
def evaluar_test(experimento):
    import matplotlib.pyplot as plt
    from sklearn.metrics import (
        classification_report, confusion_matrix, ConfusionMatrixDisplay,
        balanced_accuracy_score, f1_score,
    )

    dataset = resolver_dataset(RAIZ, RUTA_DATASET_RELATIVA)
    auditoria = verificar_auditoria(dataset)
    if auditoria["sha256_dataset"] != experimento["sha256_dataset"]:
        raise RuntimeError("El dataset es distinto al usado para entrenar.")
    clases = experimento["clases"]
    tabla_test = listar_imagenes(dataset, "test", clases=clases)
    if tabla_test.empty or set(tabla_test["clase"]) != set(clases):
        raise ValueError("Test debe contener las mismas clases del entrenamiento.")
    datos_test = crear_dataset_tf(
        tabla_test, dataset, tamano=experimento["tamano_entrada"],
        tamano_lote=TAMANO_LOTE, entrenamiento=False, semilla=SEMILLA,
    )
    probabilidades = experimento["modelo"].predict(datos_test, verbose=0)
    predicciones = np.argmax(probabilidades, axis=1)
    verdaderos = tabla_test["etiqueta"].to_numpy(dtype=np.int32)
    indices = np.arange(len(clases))
    reporte = classification_report(
        verdaderos, predicciones, labels=indices, target_names=clases,
        output_dict=True, zero_division=0,
    )
    metricas = {
        "macro_f1": float(f1_score(verdaderos, predicciones, labels=indices,
                                    average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(verdaderos, predicciones)),
    }
    carpeta = experimento["carpeta"]
    pd.DataFrame(reporte).transpose().to_csv(carpeta / "metricas_por_clase_test.csv")
    (carpeta / "metricas_test.json").write_text(
        json.dumps(metricas, indent=2), encoding="utf-8"
    )
    resultados = tabla_test[["archivo", "clase", "etiqueta"]].copy()
    resultados["prediccion"] = [clases[indice] for indice in predicciones]
    for indice, clase in enumerate(clases):
        resultados[f"probabilidad_{clase}"] = probabilidades[:, indice]
    resultados.to_csv(carpeta / "predicciones_test.csv", index=False)
    matriz = confusion_matrix(verdaderos, predicciones, labels=indices)
    grafico = ConfusionMatrixDisplay(matriz, display_labels=clases).plot(cmap="Blues")
    grafico.ax_.set_title("ResNet50 · evaluación final")
    grafico.figure_.tight_layout()
    grafico.figure_.savefig(carpeta / "matriz_confusion_test.png", dpi=150)
    plt.show()
    print(pd.DataFrame(reporte).transpose().round(4))
    print(metricas)
    return metricas, resultados


## Estado y siguiente etapa

- El EDA queda en `4. EDA/01_EDA.ipynb` y el avance actual en
  `5. feature_engineering/feature_engineering_palta.ipynb`.
- ResNet50 es la arquitectura preparada. Ejecutar las celdas de este notebook no
  descarga pesos ni entrena. La prueba técnica aislada de construcción e inferencia
  se documenta en `CAMBIOS.md`; no es un experimento de clasificación del proyecto.
- Las incidencias históricas de duplicados y etiquetas siguen pendientes. Las funciones
  de entrenamiento y evaluación comprueban los reportes antes de operar.
- Antes del experimento se deben revisar la resolución de entrada, el volteo horizontal,
  la identidad del fruto/lote y el protocolo de evaluación.

La próxima etapa utilizaría explícitamente `entrenar_resnet50`, `graficar_curvas` y,
con el modelo ya seleccionado, `evaluar_test`. No hay llamadas a esas funciones en
las celdas ejecutables de este avance.
